# Truck Stop Geocoder — Parallel (AWS Location Service)

Splits the full dataset into `NUM_WORKERS` equal chunks and geocodes them concurrently. Each worker writes `results/worker-<N>-results.csv`.

**Run the cells top to bottom.**

## Step 1 — Install dependencies *(run once)*

In [ ]:
!pip install requests pandas -q

## Step 2 — Imports, API key & region

In [ ]:
import os
import re
import requests
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

API_KEY = os.environ.get("AWS_API_KEY") or os.environ.get("AWS_LOCATION_API_KEY")
REGION = os.environ.get("AWS_DEFAULT_REGION") or os.environ.get("AWS_REGION") or "us-east-2"

if not API_KEY:
    print("WARNING: AWS API Key not found in environment variables (AWS_API_KEY / AWS_LOCATION_API_KEY)")

## Step 3 — Geocoding functions

In [ ]:
BASE_URL = f"https://places.geo.{REGION}.amazonaws.com"

def call_geocode(query):
    url = f"https://places.geo.{REGION}.amazonaws.com/v2/geocode"
    r = requests.post(
        url,
        params={"key": API_KEY},
        json={"QueryText": query, "MaxResults": 1},
        timeout=10
    )
    if r.status_code == 403:
        raise ValueError(f"403 Denied: {r.text}")
    data = r.json()
    items = data.get("ResultItems", [])
    return items[0] if items else None


def call_search_text(query):
    url = f"https://places.geo.{REGION}.amazonaws.com/v2/searchText"
    r = requests.post(
        url,
        params={"key": API_KEY},
        json={"QueryText": query, "MaxResults": 1},
        timeout=10
    )
    if r.status_code == 403:
        raise ValueError(f"403 Denied: {r.text}")
    data = r.json()
    items = data.get("ResultItems", [])
    return items[0] if items else None


def extract_coords(item):
    pos = item.get("Position", [])
    if len(pos) >= 2:
        return pos[1], pos[0]  # swap: AWS gives [lng, lat]
    return None, None


def extract_state(item):
    addr = item.get("Address", {})
    region = addr.get("Region", {})
    return region.get("Code", "")  # e.g. "OH", "FL"


def extract_label(item):
    """Get formatted address label from result."""
    return (
        item.get("Label")
        or item.get("label")
        or item.get("Place", {}).get("Label", "")
        or ""
    )


def clean_address(address):
    """Simplify highway address — keep first route reference only."""
    address = re.split(r"&", address)[0].strip()
    address = re.sub(r",\s*", " ", address)
    return address


def build_queries(row):
    """Return queries from most specific to least specific."""
    name        = row["Truckstop_Name"]
    address     = row["Address"]
    city        = row["City"]
    state       = row["State"]
    simple_addr = clean_address(address)
    return [
        f"{name}, {address}, {city}, {state}",
        f"{name}, {simple_addr}, {city}, {state}",
        f"{name}, {city}, {state}",
        f"{simple_addr}, {city}, {state}",
        f"{city}, {state}",
    ]


def geocode_row(row):
    """Try Geocode then SearchText with fallback queries; validate state."""
    queries        = build_queries(row)
    expected_state = row["State"].upper()

    for attempt, query in enumerate(queries, 1):
        # Try Geocode first
        item = call_geocode(query)
        time.sleep(0.1)

        # If Geocode returns nothing, try SearchText
        if item is None:
            item = call_search_text(query)
            time.sleep(0.1)

        if item is None:
            continue

        state = extract_state(item).upper()
        if expected_state in state or state in expected_state:
            lat, lng = extract_coords(item)
            return {
                "lat":             lat,
                "lng":             lng,
                "matched_address": extract_label(item),
                "query_used":      query,
                "attempt":         attempt,
                "status":          "OK",
            }

    # All queries exhausted — return best guess with warning
    item = call_geocode(queries[0]) or call_search_text(queries[0])
    if item:
        lat, lng = extract_coords(item)
        return {
            "lat":             lat,
            "lng":             lng,
            "matched_address": extract_label(item),
            "query_used":      queries[0],
            "attempt":         99,
            "status":          "STATE_MISMATCH",
        }

    return {
        "lat": None, "lng": None,
        "matched_address": "Not found",
        "query_used": queries[0],
        "attempt": 99,
        "status": "NOT_FOUND",
    }

## Step 4 — Run parallel workers

Writes one CSV per worker into `results/`, **saving each row as it finishes** so a crash never loses progress.
- Live progress + ETA prints every 25 rows.
- `RESUME = True` -> re-running skips rows already saved.
- Rows that return no coordinates are **kept** and flagged `Done = "not done"`.

In [ ]:
import os
import math
import csv
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

NUM_WORKERS = 20
RESULTS_DIR = "results"
RESUME = True          # True = skip rows already saved from a previous run

os.makedirs(RESULTS_DIR, exist_ok=True)

# Load the FULL dataset (not df.head()) so the split is meaningful.
work_df = pd.read_csv("fuel-prices-with-coords.csv")
total = len(work_df)
print(f"Loaded {total} rows to geocode across {NUM_WORKERS} workers\n")


def split_dataframe(df, n_workers):
    """Split df into n_workers equal, contiguous chunks: worker 1 -> [0:chunk), etc."""
    chunk_size = math.ceil(len(df) / n_workers)
    chunks = []
    for w in range(n_workers):
        start = w * chunk_size
        end = min(start + chunk_size, len(df))
        if start >= len(df):
            break
        chunks.append((w + 1, df.iloc[start:end]))
    return chunks


# ---------- shared live progress ----------
print_lock = threading.Lock()
progress_lock = threading.Lock()
completed = 0
start_time = time.time()

def bump_progress():
    """Count one processed row and print a progress line every 25 rows."""
    global completed
    with progress_lock:
        completed += 1
        c = completed
    if c % 25 == 0 or c == total:
        elapsed = time.time() - start_time
        rate = c / elapsed if elapsed else 0
        eta_min = (total - c) / rate / 60 if rate else 0
        with print_lock:
            print(f"[{c:>5}/{total}] {c/total*100:5.1f}% | "
                  f"{rate:4.1f} rows/s | elapsed {elapsed/60:4.1f} min | ETA {eta_min:4.1f} min")


def load_done_indices(out_path):
    """Row indices already saved in this worker's file (used for resume)."""
    if RESUME and os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        try:
            return set(pd.read_csv(out_path)["row_index"].tolist())
        except Exception:
            return set()
    return set()


def process_chunk(worker_id, chunk_df):
    """Geocode this worker's chunk, saving each row to disk as it completes."""
    out_path = os.path.join(RESULTS_DIR, f"worker-{worker_id}-results.csv")
    done = load_done_indices(out_path)
    file_has_data = os.path.exists(out_path) and os.path.getsize(out_path) > 0

    f = open(out_path, "a", newline="", encoding="utf-8")
    writer = None
    saved = 0

    for i, row in chunk_df.iterrows():
        if i in done:               # already geocoded in a previous run -> skip
            bump_progress()
            continue

        try:
            res = geocode_row(row)
        except Exception as e:      # network/API error on this row -> record & move on
            res = {"lat": None, "lng": None, "matched_address": f"ERROR: {e}",
                   "query_used": "", "attempt": 99, "status": "ERROR"}

        has_coords = res["lat"] is not None and res["lng"] is not None

        rec = row.to_dict()
        rec.update({
            "row_index":       i,
            "Geocoded_Lat":    res["lat"],
            "Geocoded_Lon":    res["lng"],
            "Matched_Address": res["matched_address"],
            "Query_Used":      res["query_used"],
            "Attempt":         res["attempt"],
            "Geocode_Status":  res["status"],
            "Done":            "done" if has_coords else "not done",
        })

        if writer is None:
            writer = csv.DictWriter(f, fieldnames=list(rec.keys()))
            if not file_has_data:
                writer.writeheader()
        writer.writerow(rec)
        f.flush()                   # persist immediately so a crash can't lose progress
        saved += 1
        bump_progress()

    f.close()
    with print_lock:
        print(f"  Worker {worker_id:>2} finished: +{saved} new rows -> {out_path}")
    return worker_id, out_path


chunks = split_dataframe(work_df, NUM_WORKERS)
print("Chunk assignment:")
for wid, chunk in chunks:
    print(f"  Worker {wid:>2}: rows {chunk.index[0]}-{chunk.index[-1]} ({len(chunk)} rows)")
print("\nStarting... (progress prints every 25 rows)\n")

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(process_chunk, wid, chunk): wid for wid, chunk in chunks}
    for fut in as_completed(futures):
        fut.result()               # re-raise anything that escaped a worker

print(f"\nAll workers done in {(time.time()-start_time)/60:.1f} min. CSVs in ./{RESULTS_DIR}/")

## Step 5 — Merge worker CSVs into one ordered file

In [ ]:
# Combine all worker CSVs back into one, restored to the original row order.
import glob

worker_files = sorted(
    glob.glob(os.path.join(RESULTS_DIR, "worker-*-results.csv")),
    key=lambda p: int(re.search(r"worker-(\d+)-", p).group(1)),
)

combined = pd.concat((pd.read_csv(f) for f in worker_files), ignore_index=True)
combined = (combined.drop_duplicates("row_index")
                    .sort_values("row_index")
                    .reset_index(drop=True))

combined.to_csv("truck_stops_aws_geocoded_combined.csv", index=False)
print(f"Merged {len(worker_files)} worker files -> {len(combined)} rows "
      f"in truck_stops_aws_geocoded_combined.csv")

print("\n=== Done status ===")
print(combined["Done"].value_counts())

print("\n=== Geocode status ===")
print(combined["Geocode_Status"].value_counts())

# Rows with no coordinates (kept, flagged 'not done')
not_done = combined[combined["Done"] == "not done"]
print(f"\n{len(not_done)} rows have no coordinates (Done = 'not done').")

In [ ]:
result  = pd.read_csv("truck_stops_aws_geocoded_combined.csv")


In [ ]:
result[result['Done'] =='not done']

## Step 6 — Reprocess rows with no coordinates
Re-runs geocoding on the `Done == "not done"` rows (mostly transient `ERROR`s).

In [ ]:
# Reprocess only the rows with no coordinates (Done == "not done").
import time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

COMBINED_PATH = "truck_stops_aws_geocoded_combined.csv"
RETRY_WORKERS = 10          # lower than the main run to avoid re-throttling

combined = pd.read_csv(COMBINED_PATH)
todo = combined[combined["Done"] == "not done"].copy()
n_todo = len(todo)
print(f"{n_todo} rows to reprocess\n")

retry_lock = threading.Lock()
done_count = 0
start = time.time()

def retry_row(idx_row):
    _, row = idx_row
    try:
        res = geocode_row(row)
    except Exception as e:
        res = {"lat": None, "lng": None, "matched_address": f"ERROR: {e}",
               "query_used": "", "attempt": 99, "status": "ERROR"}
    has_coords = res["lat"] is not None and res["lng"] is not None
    rec = {
        "row_index":       row["row_index"],
        "Geocoded_Lat":    res["lat"],
        "Geocoded_Lon":    res["lng"],
        "Matched_Address": res["matched_address"],
        "Query_Used":      res["query_used"],
        "Attempt":         res["attempt"],
        "Geocode_Status":  res["status"],
        "Done":            "done" if has_coords else "not done",
    }
    global done_count
    with retry_lock:
        done_count += 1
        c = done_count
    if c % 20 == 0 or c == n_todo:
        el = time.time() - start
        print(f"[{c:>4}/{n_todo}] {c/n_todo*100:5.1f}% | {c/el:4.1f} rows/s | {el:5.0f}s")
    return rec

records = []
with ThreadPoolExecutor(max_workers=RETRY_WORKERS) as ex:
    futures = [ex.submit(retry_row, ir) for ir in todo.iterrows()]
    for f in as_completed(futures):
        records.append(f.result())

retry_df = pd.DataFrame(records)
retry_df.to_csv("results/retry-results.csv", index=False)

recovered = int((retry_df["Done"] == "done").sum())
print(f"\nReprocessed {len(retry_df)} rows -> recovered {recovered}, "
      f"still not done {len(retry_df) - recovered}")
print("Saved to results/retry-results.csv")

## Step 7 — Merge recovered rows back into the combined CSV

In [ ]:
# Merge the recovered coordinates back into the combined CSV (in place).
COMBINED_PATH = "truck_stops_aws_geocoded_combined.csv"

combined = pd.read_csv(COMBINED_PATH).set_index("row_index")
retry_df = pd.read_csv("results/retry-results.csv").set_index("row_index")

update_cols = ["Geocoded_Lat", "Geocoded_Lon", "Matched_Address",
               "Query_Used", "Attempt", "Geocode_Status", "Done"]

# only overwrite rows the retry actually fixed; still-failing rows stay "not done"
fixed = retry_df[retry_df["Done"] == "done"]
combined.loc[fixed.index, update_cols] = fixed[update_cols]

combined = combined.reset_index()
combined.to_csv(COMBINED_PATH, index=False)

print(f"Merged {len(fixed)} recovered rows into {COMBINED_PATH}\n")
print("=== Done status (after merge) ===")
print(combined["Done"].value_counts())